AZURE DATABRICKS – FINAL
ASSESSMENT

Build a complete data pipeline using:
PySpark + SQL + Delta Lake + Incremental Load + DLT + Unity Catalog

PART 1 — DATAFRAME (PYSPARK)
Dataset (Given)

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

Tasks

1. Create DataFrame

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Hospital System") .getOrCreate()
df=spark.createDataFrame(data,columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



2. Add required derived columns

In [0]:
df=df.withColumn("total bill",col("consultation_fee")+col("tests_count")*500)
df.show()

+--------+------------+---------+-----------+----------------+-----------+----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total bill|
+--------+------------+---------+-----------+----------------+-----------+----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|      2500|
+--------+------------+---------+-----------+---------

3. Filter high-value patients

In [0]:
df.filter(col("total bill") > 5000).show()

+--------+------------+---------+----------+----------------+-----------+----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total bill|
+--------+------------+---------+----------+----------------+-----------+----------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|      5500|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|      5500|
+--------+------------+---------+----------+----------------+-----------+----------+



4. Perform aggregation by department

In [0]:
df.groupBy("department") \
    .agg(
        count("*").alias("patient_count"),
        sum("total bill").alias("total_revenue")
    ) \
    .show()

+-----------+-------------+-------------+
| department|patient_count|total_revenue|
+-----------+-------------+-------------+
| Cardiology|            3|        17000|
|Orthopedics|            2|         8500|
|Dermatology|            2|         4500|
|  Neurology|            1|         7500|
+-----------+-------------+-------------+



5. Sort results appropriately

In [0]:
df.orderBy(col("total bill").desc()).show()

+--------+------------+---------+-----------+----------------+-----------+----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total bill|
+--------+------------+---------+-----------+----------------+-----------+----------+
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|      2500|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|
+--------+------------+---------+-----------+---------

PART 2 — SPARK SQL

Tasks

1. Convert DataFrame to a temp view


In [0]:
df.createOrReplaceTempView("patients")

2. Write SQL to:

Fetch specific department records

Calculate revenue per city

Identify top patients

Count patients per department

In [0]:
%sql
SELECT * FROM patients WHERE department = 'Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,total bill
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500
104,Priya Nair,Bangalore,Cardiology,5000,2,6000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500


In [0]:
%sql
SELECT city, SUM(consultation_fee + tests_count*500) AS revenue
FROM patients
GROUP BY city;

city,revenue
Hyderabad,5500
Delhi,4000
Mumbai,2000
Bangalore,8500
Chennai,7500
Kolkata,4500
Ahmedabad,5500


In [0]:
%sql
SELECT patient_name, (consultation_fee + tests_count*500) AS total_cost
FROM patients
ORDER BY total_cost DESC
LIMIT 3;

patient_name,total_cost
Vikram Singh,7500
Priya Nair,6000
Arjun Reddy,5500


In [0]:
%sql
SELECT department, COUNT(*) AS patient_count
FROM patients
GROUP BY department;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


PART 3 — DELTA LAKE (CORE
OPERATIONS)

Tasks

1. Create a Delta table from the dataset

In [0]:
%sql
CREATE OR REPLACE TABLE patients_delta (
  visit_id INT,
  patient_name STRING,
  city STRING,
  department STRING,
  consultation_fee INT,
  tests_count INT
)
USING DELTA;

2. Insert new records


In [0]:
%sql
INSERT INTO patients_delta VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1);

num_affected_rows,num_inserted_rows
3,3


3. Update existing records


In [0]:
%sql
UPDATE patients_delta
SET consultation_fee = 6000
WHERE visit_id = 101;

num_affected_rows
1


4. Delete specific records


In [0]:

%sql
DELETE FROM patients_delta
WHERE visit_id = 103;

num_affected_rows
1


5. Perform an UPSERT using MERGE

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW new_patients AS
SELECT * FROM VALUES
(102,"Sneha Kapoor","Delhi","Orthopedics",3500,3),
(110,"New Patient","Chennai","Neurology",4000,1)
AS new_patients(visit_id,patient_name,city,department,consultation_fee,tests_count);

In [0]:
%sql
MERGE INTO patients_delta AS target
USING new_patients AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count

WHEN NOT MATCHED THEN
INSERT (
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count
)
VALUES (
source.visit_id,
source.patient_name,
source.city,
source.department,
source.consultation_fee,   
source.tests_count
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1
